# 📘 Solucionario ampliado — Ejercicios 2.2 a 2.5 (Pandas)

**Curso de Inteligencia Artificial — Samsung Innovation Campus**
**Capítulo 3: Análisis de datos exploratorios — Unidad 2: Pandas**

Este notebook retoma los **4 ejercicios** propuestos en `clase_19_20_21_pandas.ipynb` y desarrolla una **resolución más amplia** de cada uno: no solo el código que resuelve la consigna, sino también:

- el **razonamiento** detrás de cada paso,
- verificaciones intermedias (`display`, `print`, chequeos de forma/tipo),
- observaciones sobre **qué pasaría si algo saliera distinto**, y
- alguna herramienta extra de pandas relacionada, para ir un poco más allá de lo mínimo pedido.

Cada sección es **autocontenida**: vuelve a armar los DataFrames de base que necesita, así se puede correr una sección sola sin depender de las anteriores.

In [ ]:
# Librerías y configuración (igual que en el notebook original)
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.2
numpy : 2.4.4


---

## ✏️ Ejercicio 2.2 — Series de Pandas y DataFrames

**Consigna:**
1. Creá un `DataFrame` llamado `productos` con las columnas `nombre`, `precio` y `stock`, con al menos 4 productos inventados.
2. Agregá una nueva columna llamada `precio_con_iva` que sea `precio * 1.21`.
3. Renombrá la columna `stock` a `unidades_disponibles` **sin** modificar el DataFrame original (o sea, sin `inplace=True`).
4. Mostrá solo los productos cuyo `precio` sea mayor a 1000.

### Paso 1 — Crear el DataFrame

Partimos de un **diccionario de listas**: cada clave es el nombre de una columna, y cada lista son los valores fila por fila. Es la forma más común de armar un `DataFrame` "a mano".

In [ ]:
productos = pd.DataFrame({
    "nombre": ["Mouse", "Teclado", "Monitor", "Notebook", "Auriculares"],
    "precio": [3500, 8200, 95000, 650000, 12500],
    "stock":  [50, 30, 12, 5, 20],
})

print("Forma (filas, columnas):", productos.shape)
print()
productos

Forma (filas, columnas): (5, 3)



,nombre,precio,stock
0,Mouse,3500,50
1,Teclado,8200,30
2,Monitor,95000,12
3,Notebook,650000,5
4,Auriculares,12500,20


Antes de seguir, vale la pena chequear los **tipos de dato** de cada columna con `.info()` — `nombre` es texto (`object`), y `precio`/`stock` son numéricos (`int64`), que es justo lo que necesitamos para poder multiplicar y comparar más adelante.

In [ ]:
productos.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   nombre  5 non-null      str  
 1   precio  5 non-null      int64
 2   stock   5 non-null      int64
dtypes: int64(2), str(1)
memory usage: 252.0 bytes


### Paso 2 — Agregar `precio_con_iva`

Al asignar `productos["precio_con_iva"] = productos["precio"] * 1.21`, pandas hace una **operación vectorizada**: multiplica *cada elemento* de la columna `precio` por `1.21` a la vez (no hace falta un `for`), y el resultado queda alineado fila por fila gracias al índice compartido. Esto es una de las grandes ventajas de trabajar con `Series`/`DataFrame` en vez de listas sueltas.

In [ ]:
productos["precio_con_iva"] = productos["precio"] * 1.21

# Redondeamos solo para que se vea más prolijo (no es obligatorio)
productos["precio_con_iva"] = productos["precio_con_iva"].round(2)

productos

,nombre,precio,stock,precio_con_iva
0,Mouse,3500,50,4235.0
1,Teclado,8200,30,9922.0
2,Monitor,95000,12,114950.0
3,Notebook,650000,5,786500.0
4,Auriculares,12500,20,15125.0


### Paso 3 — Renombrar `stock` sin `inplace`

`rename()` **sin** `inplace=True` (que es el comportamiento por defecto) **no toca** el DataFrame original: devuelve una copia nueva con el cambio aplicado. Para quedarnos con esa copia, hay que guardarla en una variable nueva (acá, `productos_renombrado`).

Lo confirmamos comparando las columnas de `productos` (el original) contra las de `productos_renombrado` (la copia).

In [ ]:
productos_renombrado = productos.rename(columns={"stock": "unidades_disponibles"})

print("Columnas de productos (original):        ", productos.columns.tolist())
print("Columnas de productos_renombrado (copia):", productos_renombrado.columns.tolist())
print()
print("¿El original quedó intacto?", "stock" in productos.columns)

productos_renombrado

Columnas de productos (original):         ['nombre', 'precio', 'stock', 'precio_con_iva']
Columnas de productos_renombrado (copia): ['nombre', 'precio', 'unidades_disponibles', 'precio_con_iva']

¿El original quedó intacto? True


,nombre,precio,unidades_disponibles,precio_con_iva
0,Mouse,3500,50,4235.0
1,Teclado,8200,30,9922.0
2,Monitor,95000,12,114950.0
3,Notebook,650000,5,786500.0
4,Auriculares,12500,20,15125.0


> 💡 Si en cambio hubiésemos escrito `productos.rename(columns={"stock": "unidades_disponibles"}, inplace=True)`, la columna `stock` habría desaparecido directamente de `productos` — no habría "copia" para guardar, porque `rename(..., inplace=True)` devuelve `None`.

### Paso 4 — Filtrar productos con `precio > 1000`

La expresión `productos["precio"] > 1000` no filtra nada por sí sola: genera una **`Series` booleana** (una máscara), del mismo largo que el DataFrame, con `True`/`False` fila por fila. Al usar esa máscara como índice (`productos[mascara]`), pandas se queda solo con las filas donde dio `True` — es el equivalente a un `WHERE` de SQL.

In [ ]:
mascara_precio = productos["precio"] > 1000
print("Máscara booleana:")
print(mascara_precio)
print()

productos_caros = productos[mascara_precio]
print(f"{len(productos_caros)} de {len(productos)} productos superan los $1000:")
productos_caros

Máscara booleana:
0    True
1    True
2    True
3    True
4    True
Name: precio, dtype: bool

5 de 5 productos superan los $1000:


,nombre,precio,stock,precio_con_iva
0,Mouse,3500,50,4235.0
1,Teclado,8200,30,9922.0
2,Monitor,95000,12,114950.0
3,Notebook,650000,5,786500.0
4,Auriculares,12500,20,15125.0


---

## ✏️ Ejercicio 2.3 — Fusionar y vincular DataFrames

**Consigna:**
1. Creá dos DataFrames: `empleados` (columnas `id_emp`, `nombre`, `id_depto`) y `departamentos` (columnas `id_depto`, `nombre_depto`).
2. Usá `pd.merge()` para obtener una tabla con el nombre del empleado y el nombre de su departamento.
3. ¿Qué pasa si un empleado tiene un `id_depto` que no existe en `departamentos`? Probá con `how="left"` y con `how="inner"` y compará los resultados.

### Paso 1 — Crear los dos DataFrames

A propósito dejamos un caso "roto": Oscar tiene `id_depto = 99`, que **no existe** en `departamentos` (donde solo hay 10, 20 y 30). Esto es justo lo que la consigna pide poner a prueba en el punto 3.

In [ ]:
empleados = pd.DataFrame({
    "id_emp":   [1, 2, 3, 4],
    "nombre":   ["Lucía", "Marcos", "Nadia", "Oscar"],
    "id_depto": [10, 20, 20, 99],   # 99 no existe en departamentos
})

departamentos = pd.DataFrame({
    "id_depto":     [10, 20, 30],
    "nombre_depto": ["Ventas", "Sistemas", "RRHH"],
})

print("empleados:")
display(empleados)
print("departamentos:")
display(departamentos)

empleados:


,id_emp,nombre,id_depto
0,1,Lucía,10
1,2,Marcos,20
2,3,Nadia,20
3,4,Oscar,99


departamentos:


,id_depto,nombre_depto
0,10,Ventas
1,20,Sistemas
2,30,RRHH


### Paso 2 — `merge()` para juntar nombre de empleado + nombre de departamento

Como ambos DataFrames comparten la columna `id_depto`, pandas puede usarla como **clave de unión** con `on="id_depto"`. Por defecto, `pd.merge()` hace un `how="inner"`.

In [ ]:
tabla_completa = pd.merge(empleados, departamentos, on="id_depto")
tabla_completa

,id_emp,nombre,id_depto,nombre_depto
0,1,Lucía,10,Ventas
1,2,Marcos,20,Sistemas
2,3,Nadia,20,Sistemas


### Paso 3 — Comparar `how="inner"` vs `how="left"`

- **`inner`** (intersección): solo quedan las filas cuya clave existe **en ambos** DataFrames. Como el `id_depto=99` de Oscar no está en `departamentos`, **Oscar desaparece** del resultado.
- **`left`**: se conservan **todas** las filas de `empleados` (el DataFrame "izquierdo"), tenga o no coincidencia. Oscar se mantiene, pero con `NaN` en `nombre_depto`, porque no hay ningún departamento con `id_depto=99` para completarlo.

Lo confirmamos comparando la **cantidad de filas** de cada resultado contra la cantidad original de `empleados`, y marcando explícitamente cuáles filas quedaron con `NaN`.

In [ ]:
merge_inner = pd.merge(empleados, departamentos, on="id_depto", how="inner")
merge_left  = pd.merge(empleados, departamentos, on="id_depto", how="left")

print(f"Filas en empleados (original): {len(empleados)}")
print(f"Filas en merge inner:          {len(merge_inner)}  (Oscar se pierde)")
print(f"Filas en merge left:           {len(merge_left)}  (Oscar se mantiene)")
print()

print("INNER — Oscar no aparece:")
display(merge_inner)

print("LEFT — Oscar aparece con nombre_depto = NaN:")
display(merge_left)

print("¿Qué filas quedaron con NaN tras el left join?")
display(merge_left[merge_left["nombre_depto"].isna()])

Filas en empleados (original): 4
Filas en merge inner:          3  (Oscar se pierde)
Filas en merge left:           4  (Oscar se mantiene)

INNER — Oscar no aparece:


,id_emp,nombre,id_depto,nombre_depto
0,1,Lucía,10,Ventas
1,2,Marcos,20,Sistemas
2,3,Nadia,20,Sistemas


LEFT — Oscar aparece con nombre_depto = NaN:


,id_emp,nombre,id_depto,nombre_depto
0,1,Lucía,10,Ventas
1,2,Marcos,20,Sistemas
2,3,Nadia,20,Sistemas
3,4,Oscar,99,NaN


¿Qué filas quedaron con NaN tras el left join?


,id_emp,nombre,id_depto,nombre_depto
3,4,Oscar,99,NaN


### Extra — `right` y `outer`, para completar el panorama

- **`right`**: análogo a `left` pero priorizando el DataFrame "derecho" (`departamentos`). Como acá **RRHH** (`id_depto=30`) no tiene ningún empleado asignado, aparecería con `NaN` en las columnas de `empleados`.
- **`outer`**: la **unión** de todo — combina el comportamiento de `left` y `right` a la vez (Oscar sin depto *y* RRHH sin empleados).

In [ ]:
print("RIGHT — RRHH (id_depto=30) aparece sin empleado asignado:")
display(pd.merge(empleados, departamentos, on="id_depto", how="right"))

print("OUTER — se ven ambos casos sin match a la vez:")
display(pd.merge(empleados, departamentos, on="id_depto", how="outer"))

RIGHT — RRHH (id_depto=30) aparece sin empleado asignado:


,id_emp,nombre,id_depto,nombre_depto
0,1.0,Lucía,10,Ventas
1,2.0,Marcos,20,Sistemas
2,3.0,Nadia,20,Sistemas
3,NaN,NaN,30,RRHH


OUTER — se ven ambos casos sin match a la vez:


,id_emp,nombre,id_depto,nombre_depto
0,1.0,Lucía,10,Ventas
1,2.0,Marcos,20,Sistemas
2,3.0,Nadia,20,Sistemas
3,NaN,NaN,30,RRHH
4,4.0,Oscar,99,NaN


---

## ✏️ Ejercicio 2.4 — Clasificación de DataFrames e índice múltiple (`ex_0108`)

**Consigna:**
1. A partir del DataFrame `ventas`, ordená las filas por `vendedor` (alfabético ascendente) y, dentro de cada vendedor, por `monto` descendente.
2. Armá un `MultiIndex` de filas con niveles `region` y `vendedor`, y usalo como índice de una copia de `ventas`.
3. Usando ese índice múltiple, seleccioná todas las filas de la región `"Sur"`.

### Paso 0 — Recrear `ventas`

In [ ]:
ventas = pd.DataFrame({
    "vendedor": ["Ana", "Bruno", "Caro", "Denis", "Ana"],
    "region":   ["Norte", "Sur", "Norte", "Sur", "Sur"],
    "monto":    [1500, 3200, 800, 3200, 2100],
})
ventas

,vendedor,region,monto
0,Ana,Norte,1500
1,Bruno,Sur,3200
2,Caro,Norte,800
3,Denis,Sur,3200
4,Ana,Sur,2100


### Paso 1 — Orden combinado: `vendedor` ascendente, `monto` descendente

Cuando se pasa una **lista de columnas** a `sort_values()`, pandas ordena "en cascada": primero por la primera columna, y para las filas que empatan en esa primera columna, desempata usando la segunda. La lista `ascending=[True, False]` indica el sentido de cada criterio, en el mismo orden.

Notá el caso de **Ana**, que aparece dos veces (`$1500` y `$2100`): al ordenar por `vendedor` ascendente y `monto` descendente, sus dos filas quedan juntas y con el monto mayor primero.

In [ ]:
ventas_ordenadas = ventas.sort_values(["vendedor", "monto"], ascending=[True, False])
ventas_ordenadas

,vendedor,region,monto
4,Ana,Sur,2100
0,Ana,Norte,1500
1,Bruno,Sur,3200
2,Caro,Norte,800
3,Denis,Sur,3200


### Paso 2 — `MultiIndex` de filas con `region` y `vendedor`

`set_index(["region", "vendedor"])` toma esas dos columnas y las convierte en un índice de **dos niveles**: el nivel exterior (`region`) agrupa, y el interior (`vendedor`) identifica dentro de cada grupo. Después usamos `sort_index()` para que las regiones (y, dentro de cada una, los vendedores) queden ordenados alfabéticamente — así es mucho más fácil de leer y de indexar.

Trabajamos sobre **una copia** (`ventas.set_index(...)` ya devuelve un DataFrame nuevo por defecto, sin tocar `ventas`).

In [ ]:
ventas_multi = ventas.set_index(["region", "vendedor"]).sort_index()

print("Nombres de los niveles del índice:", ventas_multi.index.names)
print("¿El 'ventas' original sigue con índice numérico?", ventas.index.tolist())
print()
ventas_multi

Nombres de los niveles del índice: ['region', 'vendedor']
¿El 'ventas' original sigue con índice numérico? [0, 1, 2, 3, 4]



monto
region vendedor       
Norte  Ana        1500
       Caro        800
Sur    Ana        2100
       Bruno      3200
       Denis      3200

### Paso 3 — Seleccionar la región `"Sur"`

Con un `MultiIndex`, `.loc["Sur"]` selecciona **todo el bloque** cuyo nivel exterior (`region`) es `"Sur"`, y automáticamente "baja" un nivel: el resultado queda indexado solo por `vendedor`.

In [ ]:
ventas_sur = ventas_multi.loc["Sur"]
ventas_sur

,monto
vendedor,
Ana,2100
Bruno,3200
Denis,3200


### Extra — Acceder a un vendedor puntual dentro de una región

Si además querés bajar hasta un vendedor específico, se puede pasar una **tupla** con los dos niveles: `.loc[("Sur", "Denis")]`. Esto devuelve directamente la fila (como `Series`), sin pasar por el paso intermedio de quedarte con toda la región.

También existe `.xs()` (*cross-section*), que sirve para "cortar" por un nivel del índice sin importar en qué posición esté — por ejemplo, quedarte con todas las filas de `"Ana"` sin importar la región.

In [ ]:
# Acceso puntual con tupla (region, vendedor)
print("Fila puntual (Sur, Denis):")
display(ventas_multi.loc[("Sur", "Denis")])

# xs() para cortar por el nivel "vendedor", sin importar la región
print('Todas las ventas de "Ana", usando xs() sobre el nivel "vendedor":')
display(ventas_multi.xs("Ana", level="vendedor"))

Fila puntual (Sur, Denis):


monto    3200
Name: (Sur, Denis), dtype: int64

Todas las ventas de "Ana", usando xs() sobre el nivel "vendedor":


,monto
region,
Norte,1500
Sur,2100


---

## ✏️ Ejercicio 2.5 — Estadísticas descriptivas, agrupamiento y muestreo

**Consigna:**
1. Con `iris_df`, agrupá por `variety` y calculá el promedio (`mean`) de `sepal_length` y `petal_length` para cada variedad.
2. Armá una `pivot_table` con `variety` como índice y el promedio de `petal_width` como valor.
3. Creá una nueva columna `sepal_length_bin` que divida `sepal_length` en 2 categorías (`"chica"`, `"grande"`) usando `pd.cut()`.
4. Tomá una muestra aleatoria de 4 filas de `iris_df` con `random_state=7`.

### Paso 0 — Recrear `iris_df`

Usamos el mismo mini-dataset "Iris" armado a mano en el notebook original, con las columnas ya renombradas (`.` → `_`).

In [ ]:
iris_df = pd.DataFrame({
    "sepal.length": [5.1, 4.9, 4.7, 7.0, 6.4, 6.9, 6.3, 5.8, 7.1, 6.5],
    "sepal.width":  [3.5, 3.0, 3.2, 3.2, 3.2, 3.1, 3.3, 2.7, 3.0, 3.0],
    "petal.length": [1.4, 1.4, 1.3, 4.7, 4.5, 4.9, 6.0, 5.1, 5.9, 5.8],
    "petal.width":  [0.2, 0.2, 0.2, 1.4, 1.5, 1.5, 2.5, 1.9, 2.1, 2.2],
    "variety":      ["Setosa", "Setosa", "Setosa", "Versicolor", "Versicolor",
                      "Versicolor", "Virginica", "Virginica", "Virginica", "Virginica"],
})
iris_df.columns = iris_df.columns.str.replace(".", "_", regex=False)
iris_df

,sepal_length,sepal_width,petal_length,petal_width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,7.0,3.2,4.7,1.4,Versicolor
4,6.4,3.2,4.5,1.5,Versicolor
5,6.9,3.1,4.9,1.5,Versicolor
6,6.3,3.3,6.0,2.5,Virginica
7,5.8,2.7,5.1,1.9,Virginica
8,7.1,3.0,5.9,2.1,Virginica
9,6.5,3.0,5.8,2.2,Virginica


### Paso 1 — `groupby("variety")` con el promedio de dos columnas

`groupby("variety")` agrupa las 10 filas en 3 grupos (una por variedad). Al aplicar `[["sepal_length", "petal_length"]].mean()`, pandas calcula el promedio **de cada columna, dentro de cada grupo**, y devuelve un DataFrame con `variety` como índice.

De yapa, agregamos también el **desvío estándar** (`std`) con `.agg(["mean", "std"])`, para ver no solo el promedio sino qué tan dispersos están los valores dentro de cada variedad.

In [ ]:
promedios = iris_df.groupby("variety")[["sepal_length", "petal_length"]].mean()
print("Promedio de sepal_length y petal_length por variedad:")
display(promedios)

print("Mismo agrupamiento, con mean Y std (desvío estándar) a la vez:")
display(iris_df.groupby("variety")[["sepal_length", "petal_length"]].agg(["mean", "std"]))

Promedio de sepal_length y petal_length por variedad:


,sepal_length,petal_length
variety,,
Setosa,4.900000,1.366667
Versicolor,6.766667,4.700000
Virginica,6.425000,5.700000


Mismo agrupamiento, con mean Y std (desvío estándar) a la vez:


sepal_length           petal_length          
                   mean       std         mean       std
variety                                                 
Setosa         4.900000  0.200000     1.366667  0.057735
Versicolor     6.766667  0.321455     4.700000  0.200000
Virginica      6.425000  0.537742     5.700000  0.408248

### Paso 2 — `pivot_table` con el promedio de `petal_width`

`pivot_table(index="variety", values="petal_width", aggfunc="mean")` es, en este caso simple (una sola columna de agrupamiento y una sola de valores), **equivalente** a hacer `groupby("variety")["petal_width"].mean()`. La diferencia real de `pivot_table` se nota cuando además se usa `columns=` para "abrir" una segunda dimensión — lo comparamos abajo para que quede clara la equivalencia.

In [ ]:
pivot_petalo = iris_df.pivot_table(index="variety", values="petal_width", aggfunc="mean")
print("pivot_table — promedio de petal_width por variedad:")
display(pivot_petalo)

groupby_petalo = iris_df.groupby("variety")["petal_width"].mean()
print("Mismo resultado calculado con groupby (para comparar):")
display(groupby_petalo)

print("¿Dan exactamente el mismo valor?",
      pivot_petalo["petal_width"].equals(groupby_petalo))

pivot_table — promedio de petal_width por variedad:

,petal_width
variety,
Setosa,0.200000
Versicolor,1.466667
Virginica,2.175000


Mismo resultado calculado con groupby (para comparar):


variety
Setosa        0.200000
Versicolor    1.466667
Virginica     2.175000
Name: petal_width, dtype: float64

¿Dan exactamente el mismo valor? True


### Paso 3 — `sepal_length_bin` con `pd.cut()` en 2 categorías

Al pasar `bins=2` (un número, en vez de una lista de límites), `pd.cut()` **calcula automáticamente** los cortes: divide el rango completo de `sepal_length` (de mínimo a máximo) en 2 intervalos de **igual ancho**, y les asigna las etiquetas `"chica"` / `"grande"` en orden. Usamos `retbins=True` para poder ver exactamente dónde quedó el corte.

In [ ]:
sepal_bins, limites_calculados = pd.cut(
    iris_df["sepal_length"], bins=2, labels=["chica", "grande"], retbins=True
)
iris_df["sepal_length_bin"] = sepal_bins

print("Límites de los bins calculados automáticamente:", limites_calculados.round(2))
print()
display(iris_df[["sepal_length", "sepal_length_bin"]])

print("Cantidad de flores por categoría:")
display(iris_df["sepal_length_bin"].value_counts())

Límites de los bins calculados automáticamente: [4.7 5.9 7.1]



,sepal_length,sepal_length_bin
0,5.1,chica
1,4.9,chica
2,4.7,chica
3,7.0,grande
4,6.4,grande
5,6.9,grande
6,6.3,grande
7,5.8,chica
8,7.1,grande
9,6.5,grande


Cantidad de flores por categoría:


sepal_length_bin
grande    6
chica     4
Name: count, dtype: int64

### Paso 4 — Muestra aleatoria reproducible con `sample()`

`random_state=7` fija la "semilla" del generador aleatorio: con la misma semilla, `sample()` siempre devuelve **la misma muestra**, aunque se vuelva a correr la celda o el notebook entero. Lo comprobamos generando la muestra dos veces y verificando que sean idénticas.

In [ ]:
muestra_1 = iris_df.sample(4, random_state=7)
muestra_2 = iris_df.sample(4, random_state=7)

print("¿Las dos muestras con random_state=7 son idénticas?", muestra_1.equals(muestra_2))
print()
muestra_1

¿Las dos muestras con random_state=7 son idénticas? True



,sepal_length,sepal_width,petal_length,petal_width,variety,sepal_length_bin
8,7.1,3.0,5.9,2.1,Virginica,grande
5,6.9,3.1,4.9,1.5,Versicolor,grande
0,5.1,3.5,1.4,0.2,Setosa,chica
2,4.7,3.2,1.3,0.2,Setosa,chica


---

## 🎯 Cierre

Además de resolver lo pedido en cada ejercicio, en esta versión ampliada quedaron algunas herramientas extra que suelen ser útiles en la práctica:

| Ejercicio | Herramienta extra usada | Para qué sirve |
|---|---|---|
| 2.2 | máscara booleana explícita | Entender qué hay "detrás" de `df[condición]` |
| 2.3 | `.isna()`, `how="right"`, `how="outer"` | Detectar y explorar los distintos tipos de `merge` |
| 2.4 | `.loc[(nivel1, nivel2)]`, `.xs()` | Navegar un `MultiIndex` de más de una forma |
| 2.5 | `.agg(["mean","std"])`, `retbins=True` | Sacarle más jugo a `groupby` y `pd.cut` |

